## 3_2 Training using SVM Linear

In [2]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix , PrecisionRecallDisplay ,balanced_accuracy_score, matthews_corrcoef
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, KFold
import numpy as np
from sklearn.model_selection import RandomizedSearchCV , learning_curve
from scipy.stats import uniform
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv('ds/Fraud_cleaned_dataset.csv' )
data.head(1)

,Unnamed: 0,category,amt,gender,lat,long,city_pop,job,merch_lat,merch_long,is_fraud,trans_hour,trans_month,trans_day,trans_year,age
0,0,8,-0.407826,0,-0.48442,0.65762,-0.282589,370,-0.494354,0.593864,0,-1.878145,-1.504564,-1.652258,-0.634065,-0.890761


In [16]:

# target
y = data['is_fraud']

# features rest columns
X = data.drop(columns=['is_fraud'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
#search best parameters

#it takes too much time to check all the parameters we choose just C the most importan
#and also class weight should be balnced because we have imbalance in clases
param_distributions = {
    'C': uniform(0.1, 100),
}
 
#C regulirastion
random_search = RandomizedSearchCV(
    estimator = SVC(kernel='rbf', max_iter=10000, class_weight='balanced'),
    param_distributions=param_distributions,
    n_iter=10,
    cv=3 , # 3k validation fold
    scoring='balanced_accuracy', # to check precision for 2 class equaly
    random_state=42,
    n_jobs=-1 
)

random_search.fit(X_train, y_train)

#!!! it takes time even on google colab need real resources it uses cpu performonce not gpu

In [ ]:
#params found by random search
print(f"parameters : {random_search.best_params_}")
print(f" balanced_accuracy : {random_search.best_score_:.4f}")

In [ ]:

#svm model linear 
svm_model = SVC(kernel='rbf'  , class_weight='balanced' , max_iter=10000)

svm_model.fit(X_train, y_train)

#prediction
y_pred = svm_model.predict(X_test)

#we need this to analyse under and over fitting
y_pred_train = svm_model.predict(X_train)

#!!! it takes time even on google colab need real resources


In [ ]:
# simple evaluation
print(classification_report(y_test, y_pred))

In [ ]:
#validation croise k fold 10 splits

kf = KFold(n_splits=10, shuffle=True, random_state=42)

scores = cross_val_score(svm_model, X, y, cv=kf, scoring='precision_macro')

print('based on precision_macro (average of both class ) ')
print(f"cross validation 10 Parts : {scores}")
print(f" Average: {np.mean(scores):.2f}")

In [ ]:
#confusion matric
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f" TP: {tp} , TN: {tn} , FP: {fp} , FN: {fn} , ")

In [ ]:
#metrics avancee

metrics_av = {
    "MCC": {
        "Train": matthews_corrcoef(y_train, y_pred_train),
        "Test": matthews_corrcoef(y_test, y_pred)
    },
    "Balanced_Accuracy": {
        "Train": balanced_accuracy_score(y_train, y_pred_train),
        "Test": balanced_accuracy_score(y_test, y_pred)
    }
}


print("mcc score learning : " , metrics_av["MCC"]["Train"])
print("mcc score test : " , metrics_av["MCC"]["Test"])
print()
print("Balnced Accuracy learning : " , metrics_av["Balanced_Accuracy"]["Train"])
print("Balnced Accuracy   test : " , metrics_av["Balanced_Accuracy"]["Test"])

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    svm_model, X, y, cv=4, scoring='balanced_accuracy',
    train_sizes=np.linspace(0.5, 1.0, 8),
    error_score=np.nan
)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.plot(train_sizes, train_mean, label='Train')
plt.plot(train_sizes, test_mean, label='Validation')
plt.xlabel('Training Size')
plt.ylabel('Balanced Accuracy')
plt.legend()
plt.show()

In [ ]:
display = PrecisionRecallDisplay.from_estimator(
    random_search.best_estimator_, X_test, y_test, name=" LinearSVC"
)
plt.title("Precision-Recall Curve")
plt.show()

In [ ]:
import joblib

# Save the model
joblib.dump(svm_model, 'parameters/rbf_svm_fraud_model.pkl')
